## 🎯 Learning Objectives
* Apply a pretrained ResNet model for image classification.
* Understand and implement image preprocessing steps required by pretrained models.
* Perform inference with a deep learning model and interpret its predictions.
* Map model output logits to human-readable class labels.


## Exercise: Classify Images with a Pretrained ResNet

### Objective
In this exercise, you will leverage the power of transfer learning by using a state-of-the-art pretrained Convolutional Neural Network (CNN) to classify images. Specifically, you will use a ResNet model, pretrained on the ImageNet dataset, to predict the categories of several sample images.

### Task
Your task is to implement the full pipeline for image classification using a pretrained ResNet model. This involves loading the model, preparing the input images, performing inference, and interpreting the results.

### Requirements
1.  **Load a Pretrained Model**: Load a `resnet50` model from `torchvision.models` with weights pretrained on ImageNet.
2.  **Image Preprocessing**: Apply the correct preprocessing transformations (resizing, cropping, normalization) to the input images as expected by the pretrained ResNet model.
3.  **Inference**: Pass the preprocessed images through the model to obtain predictions.
4.  **Prediction Interpretation**: Map the model's output logits to human-readable ImageNet class labels and identify the top-5 predicted classes for each image.
5.  **Display Results**: For each image, display the image itself along with its top-5 predicted classes and their corresponding probabilities.

### Evaluation Criteria
*   **Correct Model Usage**: The `resnet50` model must be loaded correctly with pretrained weights.
*   **Accurate Preprocessing**: Images must be transformed according to the `resnet50`'s expected input format.
*   **Successful Inference**: The model should successfully process the images and produce output logits.
*   **Correct Label Mapping**: The predicted class indices must be accurately mapped to their corresponding ImageNet class names.
*   **Clear Output**: The results for each image (image display, top-5 predictions with probabilities) should be clearly presented.
*   **Code Quality**: Your code should be well-structured, readable, and include comments where necessary.


In [ ]:
# Setup Code (Run this cell first)

import torch
import torchvision
from torchvision.models import ResNet50_Weights
from PIL import Image
import matplotlib.pyplot as plt
import requests
from io import BytesIO
import json

# Ensure reproducibility
torch.manual_seed(42)

# --- Device Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Helper Function for Displaying Images ---
def display_image(img_tensor, title=""):
    """Displays a PyTorch tensor image."""
    # Convert tensor to PIL Image for display
    # Denormalize if necessary (ResNet expects normalized input)
    # For display, we assume the input is already in [0, 1] or will be converted.
    # If the image was normalized with mean/std, we'd need to reverse it.
    # For simplicity, we'll just convert to numpy and display.
    
    # If it's a batch, take the first image
    if img_tensor.dim() == 4:
        img_tensor = img_tensor[0]
        
    # Permute from (C, H, W) to (H, W, C) for matplotlib
    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    
    # Clip values to [0, 1] in case of floating point inaccuracies or denormalization issues
    img_np = img_np.clip(0, 1)
    
    plt.imshow(img_np)
    plt.title(title)
    plt.axis('off')

# --- Load ImageNet Class Labels ---
# We'll download the ImageNet class index mapping from a common source.
# This maps numerical indices to human-readable class names.

IMAGENET_LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"

try:
    response = requests.get(IMAGENET_LABELS_URL)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    imagenet_labels = response.text.splitlines()
    print(f"Loaded {len(imagenet_labels)} ImageNet class labels.")
except requests.exceptions.RequestException as e:
    print(f"Error downloading ImageNet labels: {e}")
    print("Using a mock list of labels. Please check your internet connection.")
    # Fallback to a mock list if download fails
    imagenet_labels = [f"class_{i}" for i in range(1000)]

# --- Sample Images for Classification ---
# We'll use a few diverse images for demonstration.

sample_image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b6/Felis_catus-cat_on_fence.jpg/800px-Felis_catus-cat_on_fence.jpg", # Cat
    "https://upload.wikimedia.org/wikipedia/commons/thumb/2/20/Giant_Panda_in_Beijing_Zoo_1.JPG/800px-Giant_Panda_in_Beijing_Zoo_1.JPG", # Panda
    "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f9/Pug_dog.jpg/800px-Pug_dog.jpg", # Pug
    "https://upload.wikimedia.org/wikipedia/commons/thumb/6/64/Mount_Everest_from_Kala_Patthar_-_October_2015.jpg/800px-Mount_Everest_from_Kala_Patthar_-_October_2015.jpg" # Mountain
]

# Load images into a list of PIL Image objects
raw_images = []
print("\nDownloading sample images...")
for i, url in enumerate(sample_image_urls):
    try:
        response = requests.get(url)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert('RGB')
        raw_images.append(img)
        print(f"Downloaded image {i+1}/{len(sample_image_urls)} from {url.split('/')[-1]}")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading image from {url}: {e}")
        print("Skipping this image.")

if not raw_images:
    raise RuntimeError("No images were loaded. Please check your internet connection or image URLs.")

print("\nSetup complete. You can now proceed with the exercise.")


### Your Implementation

Now it's your turn! Implement the image classification pipeline in the cell below. Follow the requirements outlined in the task description.

**Steps to consider:**
1.  **Load the `resnet50` model**: Use `torchvision.models.resnet50` and specify the `weights` argument for pretrained ImageNet weights.
2.  **Get the appropriate transforms**: The `ResNet50_Weights` object provides a convenient way to get the recommended preprocessing transforms.
3.  **Preprocess images**: Apply these transforms to each `PIL.Image` object in the `raw_images` list.
4.  **Create a batch**: Stack the preprocessed image tensors into a single batch tensor.
5.  **Move to device**: Transfer the model and the image batch to the appropriate device (CPU/GPU).
6.  **Perform inference**: Pass the batch through the model.
7.  **Calculate probabilities**: Apply `torch.nn.functional.softmax` to the model's output logits to get probabilities.
8.  **Get top predictions**: For each image, find the top-5 predicted class indices and their probabilities.
9.  **Map to labels**: Use the `imagenet_labels` list to convert indices to class names.
10. **Display results**: Iterate through the original images and their predictions, displaying each image and its top-5 classifications.


In [ ]:
# --- Reference Solution ---

# 1. Load the pretrained ResNet50 model
# Using ResNet50_Weights.IMAGENET1K_V2 for 2026-ready best available weights
weights = ResNet50_Weights.IMAGENET1K_V2
model = torchvision.models.resnet50(weights=weights)
model.eval() # Set the model to evaluation mode (disables dropout, batch norm updates)
model.to(device)

print(f"Loaded ResNet50 model with {weights.name} weights.")

# 2. Get the appropriate preprocessing transforms
# The weights object provides the recommended transforms for the model
preprocess = weights.transforms()
print("Obtained recommended preprocessing transforms.")

# 3. Preprocess images and create a batch
processed_images = []
for img in raw_images:
    processed_images.append(preprocess(img))

# Stack the list of tensors into a single batch tensor
# The batch dimension will be added automatically by torch.stack
input_batch = torch.stack(processed_images)
input_batch = input_batch.to(device)

print(f"Prepared a batch of {input_batch.shape[0]} images with shape {input_batch.shape[1:]}.")

# 4. Perform inference
with torch.no_grad(): # Disable gradient calculation for inference to save memory and speed up
    output = model(input_batch)

# 5. Calculate probabilities
probabilities = torch.nn.functional.softmax(output, dim=1)

print("Performed inference and calculated probabilities.")

# 6. Get top predictions and display results
num_top_predictions = 5

plt.figure(figsize=(15, 10))
for i in range(input_batch.shape[0]):
    plt.subplot(2, 2, i + 1) # Adjust subplot grid based on number of images
    
    # Display the original image (or a denormalized version of the processed one)
    # For simplicity, we'll display the raw PIL image here.
    # If we wanted to display the *processed* image, we'd need to reverse normalization.
    display_image(preprocess.denormalize(processed_images[i]), title=f"Original Image {i+1}")
    
    # Get top probabilities and their indices for the current image
    top_prob, top_indices = torch.topk(probabilities[i], num_top_predictions)
    
    # Prepare prediction text
    predictions_text = "Top 5 Predictions:\n"
    for j in range(num_top_predictions):
        class_idx = top_indices[j].item()
        class_name = imagenet_labels[class_idx]
        prob = top_prob[j].item() * 100
        predictions_text += f"{j+1}. {class_name}: {prob:.2f}%\n"
    
    # Add predictions as text below the image
    plt.text(0, 1.05, predictions_text, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.5))

plt.tight_layout()
plt.show()

print("\nClassification complete. Results displayed above.")
